## Описание проекта

Интернет-магазин собирает историю покупателей, проводит рассылки предложений и планирует будущие продажи. Для оптимизации процессов надо выделить пользователей, которые готовы совершить покупку в ближайшее время.

### Цель

Предсказать вероятность покупки в течение 90 дней

### Задачи

* Изучить данные
* Разработать полезные признаки
* Создать модель для классификации пользователей
* Улучшить модель и максимизировать метрику roc_auc
* Выполнить тестирование

### Данные

`apparel-purchases`

история покупок
* `client_id` - идентификатор пользователя
* `quantity` - количество товаров в заказе
* `price` цена товара
* `category_ids` - вложенные категории, к которым отнсится товар
* `date` - дата покупки
* `message_id` - идентификатор сообщения из рассылки

`apparel-target_binary`

совершит ли клиент покупку в течение следующих 90 дней
* `client_id` - идентификатор пользователя
* `target` - целевой признак

### Результат

Репозиторий на гитхабе:
* тетрадь jupyter notebook с описанием, подготовкой признаков, обучением модели и тестированием
* описание проекта и инструкция по использованию в файле README.md
* список зависимостей в файле requirements.txt

### Стэк

* python
* pandas
* sklearn

## Описание данных

### `apparel-purchases`

Данные о покупках клиентов по дням и по товарам. В каждой записи покупка определенного товара, его цена, количество штук.

В таблице есть списки идентификаторов, к каким категориям относится товар. Часто это вложенные категории (например автотовары-аксессуары-освежители), но также может включать в начале списка маркер распродажи или маркер женщинам/мужчинам.

Нумерация категорий сквозная для всех уровней, то есть 44 на второй позиции списка или на третьей – это одна и та же категория. Иногда дерево категорий обновляется, поэтому могут меняться вложенности, например `['4', '28', '44', '1594']` или `['4', '44', '1594']`. Как обработать такие случаи – можете предлагать свои варианты решения.
* `client_id` - идентификатор клиента
* `quantity` - количество единиц товара
* `price` - цена товара
* `category_ids` - идентификаторы категорий
* `date` - дата покупки
* `message_id` - идентификатор сообщения из рассылки

### `apparel-messages`

Рассылки, которые были отправлены клиентам из таблицы покупок.
* `bulk_campaign_id` - идентификатор рассылки
* `client_id` - идентификатор клиента
* `message_id` - идентификатор сообщения
* `event` - действие с сообщением (отправлено, открыто, покупка...)
* `channel` - канал рассылки
* `date` - дата действия
* `created_at` - дата-время полностью

### `target`

* `client_id` - идентификатор клиента
* `target` - клиент совершил покупку в целевом периоде

Общая база рассылок огромна, поэтому собрали для вас агрегированную по дням статистику по рассылкам. Если будете создавать на основе этой статистики дополнительные признаки, обратите внимание, что нельзя суммировать по колонкам nunique, потому что это уникальные клиенты в пределах дня, у вас нет данных, повторяются ли они в другие дни.

### `full_campaign_daily_event`

Агрегация общей базы рассылок по дням и типам событий
* `date` - дата
* `bulk_campaign_id` - идентификатор рассылки
* `count_event*` - общее количество каждого события event
* `nunique_event*` - количество уникальных client_id в каждом событии

\* в именах колонок найдете все типы событий event

### `full_campaign_daily_event_channel`

Агрегация по дням с учетом событий и каналов рассылки
* `date` - дата
* `bulk_campaign_id` - идентификатор рассылки
* `count_event*_channel*` - общее количество каждого события по каналам
* `nunique_event*_channel*` - количество уникальных client_id по событиям и каналам

\* в именах колонок есть все типы событий event и каналов рассылки channel

## Импорты библиотек

In [ ]:
import pandas as pd 
import numpy as np 

from tqdm import tqdm
tqdm.pandas()

from joblib import Parallel, delayed
from prettytable import PrettyTable
from sklearn.base import TransformerMixin, BaseEstimator
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn import set_config
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from optuna import distributions
from optuna.integration.sklearn import OptunaSearchCV
import warnings

set_config(transform_output="pandas")

warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn.pipeline')
warnings.filterwarnings('ignore', message='X does not have valid feature names')

Функция для красивого вывода

In [3]:
def show_html(text, color="#1D2227"):
    from IPython.display import HTML, display
    display(HTML(f'<span style="background-color: {color};">{text}</span>'))

Загрузка файла и вывод первых строк

In [4]:
file_names = ['model_features.csv', 
              'apparel_purchases_features.csv', 
              'apparel_messages_features.csv', 
              'full_campaign_daily_event_features.csv', 
              'apparel-target_binary.csv']

dataframes = []

for i, file_name in enumerate(file_names):
    if i != len(file_names) - 1:
        dataframe = pd.read_csv('./Данные/' + file_name, sep=';')
    else:
        dataframe = pd.read_csv('./Данные/' + file_name)
    dataframes.append(dataframe)
    show_html(file_name)
    print(dataframe.shape)
    display(dataframe.head())

(49849, 53)


,client_id,total_quantity_week,total_quantity_month,total_quantity_3_month,total_quantity_half_year,total_quantity_year,total_quantity_3_years,total_spent_week,total_spent_month,total_spent_3_month,...,purchase_positive_share_2,purchase_positive_share_3,purchase_share,common_channel,avg_actions_ratio,sum_actions_cost,avg_positive_actions_ratio,sum_positive_actions_cost,avg_negative_actions_ratio,sum_negative_actions_cost
0,1515915625468060902,0.0,0.0,0.0,0.0,0.0,7,0.0,0.0,0.0,...,2.840909,10.000000,2.824859,email,0.759644,42.115687,0.345211,62.103976,0.000586,-19.988289
1,1515915625468061003,0.0,0.0,0.0,0.0,0.0,7,0.0,0.0,0.0,...,0.602410,8.333333,0.602410,email,0.877879,16.103515,0.176781,16.103515,NaN,NaN
2,1515915625468061099,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,mixed,0.730997,7.957753,0.247532,11.876227,0.020659,-3.918475
3,1515915625468061100,2.0,2.0,2.0,2.0,2.0,2,2098.0,2098.0,2098.0,...,0.231481,0.606061,0.230415,mobile_push,0.541356,19.585162,0.337602,22.565396,0.005408,-2.980235
4,1515915625468061170,0.0,0.0,0.0,0.0,19.0,19,0.0,0.0,0.0,...,1.023891,6.000000,1.023891,mixed,0.764067,46.096103,0.378545,46.096103,NaN,NaN


(49849, 39)


,client_id,total_quantity_week,total_quantity_month,total_quantity_3_month,total_quantity_half_year,total_quantity_year,total_quantity_3_years,total_spent_week,total_spent_month,total_spent_3_month,...,best_category_level_2,best_count_level_2,best_percentage_level_2,best_category_level_3,best_count_level_3,best_percentage_level_3,best_category_level_4,best_count_level_4,best_percentage_level_4,days_after_last_purchase
0,1515915625468060902,0.0,0.0,0.0,0.0,0.0,7,0.0,0.0,0.0,...,28,4.0,57.142857,260,2.0,28.571429,420,2.0,28.571429,630
1,1515915625468061003,0.0,0.0,0.0,0.0,0.0,7,0.0,0.0,0.0,...,28,7.0,100.000000,249,7.0,100.000000,615,7.0,100.000000,408
2,1515915625468061099,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,...,28,1.0,100.000000,290,1.0,100.000000,424,1.0,100.000000,640
3,1515915625468061100,2.0,2.0,2.0,2.0,2.0,2,2098.0,2098.0,2098.0,...,27,2.0,100.000000,1828,2.0,100.000000,5717,2.0,100.000000,6
4,1515915625468061170,0.0,0.0,0.0,0.0,19.0,19,0.0,0.0,0.0,...,28,15.0,88.235294,260,12.0,70.588235,420,12.0,70.588235,244


(53329, 9)


,client_id,total_messages,positive_share_2,positive_share_3,neutral_share,purchase_positive_share_2,purchase_positive_share_3,purchase_share,common_channel
0,1515915625468060902,177,99.435028,28.248588,71.186441,2.840909,10.000000,2.824859,email
1,1515915625468061003,166,100.000000,7.228916,92.771084,0.602410,8.333333,0.602410,email
2,1515915625468061099,276,99.275362,21.376812,77.898551,0.000000,0.000000,0.000000,mixed
3,1515915625468061100,434,99.539171,38.018433,61.520737,0.231481,0.606061,0.230415,mobile_push
4,1515915625468061170,293,100.000000,17.064846,82.935154,1.023891,6.000000,1.023891,mixed


(53329, 7)


,client_id,avg_actions_ratio,sum_actions_cost,avg_positive_actions_ratio,sum_positive_actions_cost,avg_negative_actions_ratio,sum_negative_actions_cost
0,1515915625468060902,0.759644,42.115687,0.345211,62.103976,0.000586,-19.988289
1,1515915625468061003,0.877879,16.103515,0.176781,16.103515,NaN,NaN
2,1515915625468061099,0.730997,7.957753,0.247532,11.876227,0.020659,-3.918475
3,1515915625468061100,0.541356,19.585162,0.337602,22.565396,0.005408,-2.980235
4,1515915625468061170,0.764067,46.096103,0.378545,46.096103,NaN,NaN


(49849, 2)


,client_id,target
0,1515915625468060902,0
1,1515915625468061003,1
2,1515915625468061099,0
3,1515915625468061100,0
4,1515915625468061170,0


Объединяем полученные признаки и целевой признак

In [5]:
data = pd.merge(dataframes[0], dataframes[4], on='client_id', how='right')
display(data.head())

,client_id,total_quantity_week,total_quantity_month,total_quantity_3_month,total_quantity_half_year,total_quantity_year,total_quantity_3_years,total_spent_week,total_spent_month,total_spent_3_month,...,purchase_positive_share_3,purchase_share,common_channel,avg_actions_ratio,sum_actions_cost,avg_positive_actions_ratio,sum_positive_actions_cost,avg_negative_actions_ratio,sum_negative_actions_cost,target
0,1515915625468060902,0.0,0.0,0.0,0.0,0.0,7,0.0,0.0,0.0,...,10.000000,2.824859,email,0.759644,42.115687,0.345211,62.103976,0.000586,-19.988289,0
1,1515915625468061003,0.0,0.0,0.0,0.0,0.0,7,0.0,0.0,0.0,...,8.333333,0.602410,email,0.877879,16.103515,0.176781,16.103515,NaN,NaN,1
2,1515915625468061099,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,...,0.000000,0.000000,mixed,0.730997,7.957753,0.247532,11.876227,0.020659,-3.918475,0
3,1515915625468061100,2.0,2.0,2.0,2.0,2.0,2,2098.0,2098.0,2098.0,...,0.606061,0.230415,mobile_push,0.541356,19.585162,0.337602,22.565396,0.005408,-2.980235,0
4,1515915625468061170,0.0,0.0,0.0,0.0,19.0,19,0.0,0.0,0.0,...,6.000000,1.023891,mixed,0.764067,46.096103,0.378545,46.096103,NaN,NaN,0


Анализ пропусков и дупликатов

In [6]:
display(data.isna().sum())
display(data.duplicated().sum())

client_id                         0
total_quantity_week               0
total_quantity_month              0
total_quantity_3_month            0
total_quantity_half_year          0
total_quantity_year               0
total_quantity_3_years            0
total_spent_week                  0
total_spent_month                 0
total_spent_3_month               0
total_spent_half_year             0
total_spent_year                  0
total_spent_3_years               0
avg_quantity_week                 0
avg_quantity_month                0
avg_quantity_3_month              0
avg_quantity_half_year            0
avg_quantity_year                 0
avg_quantity_3_years              0
avg_quantity_per_order            0
avg_spent_week                    0
avg_spent_month                   0
avg_spent_3_month                 0
avg_spent_half_year               0
avg_spent_year                    0
avg_spent_3_years                 0
best_category_level_1             0
best_count_level_1          

0

Пропуски есть в столбцах, полученных из `apparel_messages.csv` и `full_campaign_daily_event_channel.csv`:
* Столбцы из `apparel_messages.csv`, по всей видимости, имеют пропуски одновременно, т.е. строки со всеми пропусками (Это клиенты, для которых по каким-то причинам нет информации о рассылках)
* Столбцы из `full_campaign_daily_event_channel.csv` имеют разное количество пропусков, что может быть связано с тем, что некоторые клиенты не совершали позитивных/негативных действий

#### Проверка полученных датафреймов

Проверка `client_id`

In [7]:
field_names = ["df1/df2"] + file_names
mytable = PrettyTable()
mytable.field_names = field_names

for i in range(len(dataframes)):
    df1 = dataframes[i]
    file_name1 = file_names[i]
    row = []
    row.append(file_name1)
    for j in range(len(dataframes)):
        df2 = dataframes[j]
        file_name2 = file_names[j]
        ids1 = df1['client_id'].unique()
        ids2 = df2['client_id'].unique()
        intersection = np.intersect1d(ids1, ids2)
        row.append((len(ids1), len(intersection), len(ids2)))
    mytable.add_row(row)
print(mytable)

+----------------------------------------+-----------------------+--------------------------------+-------------------------------+----------------------------------------+---------------------------+
|                df1/df2                 |   model_features.csv  | apparel_purchases_features.csv | apparel_messages_features.csv | full_campaign_daily_event_features.csv | apparel-target_binary.csv |
+----------------------------------------+-----------------------+--------------------------------+-------------------------------+----------------------------------------+---------------------------+
|           model_features.csv           | (49849, 49849, 49849) |     (49849, 49849, 49849)      |     (49849, 41982, 53329)     |         (49849, 41982, 53329)          |   (49849, 49849, 49849)   |
|     apparel_purchases_features.csv     | (49849, 49849, 49849) |     (49849, 49849, 49849)      |     (49849, 41982, 53329)     |         (49849, 41982, 53329)          |   (49849, 49849, 49849)

Для `7 867` строк нет информации из таблиц `apparel_messages.csv` и `full_campaign_daily_features.csv`. В этих строках есть информация о покупках, так что удалять их не можем. 

Заполним пропуски `нулями` везде, где это подходит по смыслу. Исключения:
* `common_channel` - особое значение `unknown`
* `positive_share_2`, `positive_share_3` - заполняем значением `1`, т.к. информации о других действиях нет
* `purchase_positive_share_3`, `purchase_share` - также заполняем значением `1`, т.к. есть информация только о покупках




Поля, пропуски в которых требуется заполнить

In [8]:
nan_top = data.isna().sum().sort_values(ascending=False)
columns_with_nan = nan_top[nan_top > 0].index.tolist()
columns_with_nan

['sum_negative_actions_cost',
 'avg_negative_actions_ratio',
 'sum_positive_actions_cost',
 'avg_positive_actions_ratio',
 'positive_share_3',
 'purchase_positive_share_2',
 'total_messages',
 'positive_share_2',
 'neutral_share',
 'purchase_positive_share_3',
 'purchase_share',
 'common_channel',
 'avg_actions_ratio',
 'sum_actions_cost']

In [9]:
data['common_channel'] = data['common_channel'].fillna('unknown')
columns_to_fill_1 = ['positive_share_2', 'positive_share_3', 'purchase_positive_share_3', 'purchase_share']
data[columns_to_fill_1] = data[columns_to_fill_1].fillna(1)
filled_columns = columns_to_fill_1 + ['common_channel']
columns_to_fill_0 = [column for column in columns_with_nan if column not in filled_columns]
data[columns_to_fill_0] = data[columns_to_fill_0].fillna(0)


Проверка

In [10]:
data.isna().sum().sort_values(ascending=False)

client_id                     0
positive_share_2              0
best_category_level_2         0
best_count_level_2            0
best_percentage_level_2       0
best_category_level_3         0
best_count_level_3            0
best_percentage_level_3       0
best_category_level_4         0
best_count_level_4            0
best_percentage_level_4       0
days_after_last_purchase      0
total_messages                0
positive_share_3              0
total_quantity_week           0
neutral_share                 0
purchase_positive_share_2     0
purchase_positive_share_3     0
purchase_share                0
common_channel                0
avg_actions_ratio             0
sum_actions_cost              0
avg_positive_actions_ratio    0
sum_positive_actions_cost     0
avg_negative_actions_ratio    0
sum_negative_actions_cost     0
best_percentage_level_1       0
best_count_level_1            0
best_category_level_1         0
avg_spent_3_years             0
total_quantity_month          0
total_qu

Имеем только одно категориальное поле, все остальные - числовые непрерывные

#### Разделение данных

In [12]:
RANDOM_STATE = 42

In [13]:
X = data.drop(columns=['client_id', 'target'])
y = data['target']

In [14]:
X_train, X_test, y_train, y_test = \
    train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

Энкодер для кодирования поля `common_channel`

In [15]:
class CategoryEncoder(TransformerMixin, BaseEstimator):
    """Трансформер для приведения категориальных колонок к типу 'category'"""
    def __init__(self, columns_to_convert=None, values_dict={}, columns=[]):
        self.columns_to_convert = columns_to_convert
        self.values_dict = values_dict
        self.columns = columns

    def fit(self, X, y=None):
        # Проверяем, что колонки существуют
        if isinstance(X, pd.DataFrame):
            for column in self.columns_to_convert:
                if column not in self.columns:
                    raise ValueError(f"Column {column} not found in data")
        self._is_fitted = True
        return self

    def transform(self, X):
        X_copy = X.copy()
        if isinstance(X, np.ndarray):
            convert_columns_idx = [self.columns.index(column) for column in self.columns_to_convert if column in self.columns]
            for idx in convert_columns_idx:
                X_copy[:, idx] = np.array([self.values_dict.get(x, -1) for x in X_copy[:, idx]])
        elif isinstance(X, pd.DataFrame):
            for column in self.columns_to_convert:
                if column in X_copy.columns:
                    X_copy[column] = X_copy[column]\
                        .apply(lambda x: self.values_dict.get(x, -1))
        return X_copy
    
    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.columns, dtype=object)

In [16]:
numerical_columns_0 = columns_to_fill_0
numerical_columns_1 = columns_to_fill_1
categorical_columns = ['common_channel']
categorical_columns_indexes = [data.columns.get_loc(column) for column in categorical_columns]

In [17]:
numerical_pipe_0 = Pipeline([
    ('simple_imputer', SimpleImputer(strategy='constant', fill_value=1))
])
numerical_pipe_1 = Pipeline([
    ('simple_imputer', SimpleImputer(strategy='constant', fill_value=0))
])
categorical_pipe = Pipeline([
    ('category_encoder', CategoryEncoder(columns_to_convert=categorical_columns, 
                                         values_dict={'unknown': 0,
                                                      'email': 1,
                                                      'mobile_push': 2,
                                                      'mixed': 1.5},
                                         columns=categorical_columns))
])

preprocessor = ColumnTransformer([
    ('numerical_0', numerical_pipe_0, numerical_columns_0),
    ('numerical_1', numerical_pipe_1, numerical_columns_1),
    ('categorical', categorical_pipe, categorical_columns)
],
    remainder='drop',
    verbose_feature_names_out=False
)

In [ ]:
pipe_parameters = {
    # LightGBM
    "LightGBM": (
        "LightGBM",
        Pipeline([
            ('prep', preprocessor),
            ('model', LGBMClassifier(
                num_leaves=31,
                max_depth=-1,
                learning_rate=0.1,
                n_estimators=400,
                n_jobs=-1,
                verbose=-1
            ))
        ]),
        {
            'model__num_leaves': distributions.IntDistribution(10, 60),
            'model__max_depth': distributions.IntDistribution(3, 12),
            'model__learning_rate': distributions.FloatDistribution(0.02, 0.2, log=True),
            'model__min_child_samples': distributions.IntDistribution(20, 100)
        }
    ),
    
    # CatBoost
    "CatBoost": (
        "CatBoost",
        Pipeline([
            ('prep', preprocessor),
            ('model', CatBoostClassifier(
                iterations=400,
                random_seed=RANDOM_STATE,
                verbose=False,
                task_type='CPU',  # Отключить GPU для избежания NaN ошибок
                thread_count=4
            ))
        ]),
        {
            'model__learning_rate': distributions.FloatDistribution(0.02, 0.2, log=True),
            'model__depth': distributions.IntDistribution(3, 12),
            'model__l2_leaf_reg': distributions.FloatDistribution(1, 50, log=True),
            'model__bagging_temperature': distributions.FloatDistribution(0, 1),
            'model__random_strength': distributions.FloatDistribution(0, 2)
        }
    )
}

# Явно указать cv с StratifiedKFold для балансировки классов
cv_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

Первые строки таблицы `X_train`

In [19]:
X_train.head()

,total_quantity_week,total_quantity_month,total_quantity_3_month,total_quantity_half_year,total_quantity_year,total_quantity_3_years,total_spent_week,total_spent_month,total_spent_3_month,total_spent_half_year,...,purchase_positive_share_2,purchase_positive_share_3,purchase_share,common_channel,avg_actions_ratio,sum_actions_cost,avg_positive_actions_ratio,sum_positive_actions_cost,avg_negative_actions_ratio,sum_negative_actions_cost
49433,0.0,2.0,2.0,2.0,2.0,2,0.0,3498.0,3498.0,3498.0,...,1.315789,4.000000,1.298701,mobile_push,0.553848,11.720889,0.332365,13.717504,0.001694,-1.996615
41537,0.0,0.0,0.0,0.0,0.0,3,0.0,0.0,0.0,0.0,...,1.704545,2.752294,1.694915,email,0.536696,78.271745,0.302261,80.267202,0.002274,-1.995457
17439,0.0,0.0,0.0,0.0,2.0,2,0.0,0.0,0.0,0.0,...,0.537634,6.250000,0.537634,email,0.876622,15.589608,0.257229,15.589608,0.000000,0.000000
27245,0.0,0.0,0.0,0.0,2.0,2,0.0,0.0,0.0,0.0,...,1.515152,14.285714,1.515152,email,0.883145,11.222745,0.449404,11.222745,0.000000,0.000000
18836,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,...,0.309598,6.250000,0.309598,mixed,0.785605,14.574060,0.419976,14.574060,0.000000,0.000000


In [20]:
best_models = {}

# Подавить RuntimeWarnings от numpy/optuna при cv валидации
warnings.filterwarnings('ignore', category=RuntimeWarning, module='numpy')

for model_name, (model_display_name, model_pipeline, param_distributions) in pipe_parameters.items():
    results = OptunaSearchCV(
        estimator=model_pipeline,
        param_distributions=param_distributions,
        n_trials=100,
        cv=cv_split,
        scoring='roc_auc',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        error_score='raise'  # Явно обрабатывать ошибки, а не молча заменять на NaN
    )
    try:
        results.fit(X_train, y_train)
        best_models[model_name] = (model_display_name, results.best_estimator_, results.best_score_)
    except Exception as e:
        print(f"Ошибка при обучении {model_name}: {e}")
        best_models[model_name] = None

C:\Temp\ipykernel_29724\2523481692.py:7: ExperimentalWarning: OptunaSearchCV is experimental (supported from v0.17.0). The interface can change in the future.
  results = OptunaSearchCV(
[I 2026-04-02 19:33:35,032] A new study created in memory with name: no-name-829e6d3f-7651-46ca-aa56-378d14466674
[I 2026-04-02 19:33:45,778] Trial 7 finished with value: 0.6810985210119627 and parameters: {'model__num_leaves': 34, 'model__max_depth': 3, 'model__learning_rate': 0.08582307758641836, 'model__min_child_samples': 89}. Best is trial 7 with value: 0.6810985210119627.
[I 2026-04-02 19:33:46,180] Trial 19 finished with value: 0.6895287523902892 and parameters: {'model__num_leaves': 57, 'model__max_depth': 3, 'model__learning_rate': 0.05067847905700451, 'model__min_child_samples': 37}. Best is trial 19 with value: 0.6895287523902892.
[I 2026-04-02 19:33:49,895] Trial 13 finished with value: 0.6882725080455463 and parameters: {'model__num_leaves': 45, 'model__max_depth': 4, 'model__learning_rate

In [25]:
best_model_params_dict = best_models['CatBoost'][1].named_steps['model'].get_params()
best_model_params = {}
for key in best_model_params_dict.keys():
    best_model_params['model' + '__' + key] = best_model_params_dict[key]
print(best_model_params)

{'model__iterations': 400, 'model__thread_count': 4, 'model__random_seed': 42, 'model__verbose': False, 'model__task_type': 'CPU', 'model__learning_rate': 0.034027366561166714, 'model__depth': 3, 'model__l2_leaf_reg': 34.32934866718578, 'model__bagging_temperature': 0.5197659972824272, 'model__random_strength': 1.4207115010314675}


Лучшая модель - `CatBoost` с параметрами:
* 'model__iterations': 400, 
* 'model__thread_count': 4, 
* 'model__random_seed': 42, 
* 'model__verbose': False, 
* 'model__task_type': 'CPU', 
* 'model__learning_rate': 0.034027366561166714, 
* 'model__depth': 3, 
* 'model__l2_leaf_reg': 34.32934866718578, 
* 'model__bagging_temperature': 0.5197659972824272, 
* 'model__random_strength': 1.4207115010314675

#### Предсказание на тестовой выборке

In [33]:
best_pipeline = clone(pipe_parameters["CatBoost"][1])
best_pipeline.set_params(**best_model_params)

best_pipeline.fit(X_train, y_train)

y_test_probas = best_pipeline.predict_proba(X_test)[:, 1]
roc_auc_test = roc_auc_score(y_test, y_test_probas)

print('Значение метрики roc-auc на тестовой выборке для лучшей модели =', roc_auc_test)

Значение метрики roc-auc на тестовой выборке для лучшей модели = 0.7042406529112974
